In [14]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertModel, BertTokenizer
from torchvision import models, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm.auto import tqdm
import os
import torch.nn.functional as F  # 核心修复：添加这一行


LOCAL_BERT_PATH = "/mnt/workspace/FRD-CLIP/models/bert-base-chinese"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LIIMR_Baseline(nn.Module):
    """
    LIIMR 基础框架：简单的后期融合（Late Fusion）
    特征处理逻辑：
    1. BERT 提取文本特征 (768维)
    2. ResNet 提取全局图像特征 (2048维)
    3. 简单的 Concat 拼接
    4. 线性层直接分类
    """
    def __init__(self, bert_name="bert-base-chinese", num_classes=2):
        super().__init__()
        
        # 1. 文本分支：纯 BERT
        self.bert = BertModel.from_pretrained(bert_name)
        # 冻结部分权重以模拟基础 Baseline 的常规操作
        for param in self.bert.parameters():
            param.requires_grad = False
        # 仅解冻 Pooler 层
        for param in self.bert.pooler.parameters():
            param.requires_grad = True

        # 2. 图像分支：纯 ResNet
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.visual_backbone = nn.Sequential(*list(resnet.children())[:-1]) # 取到 AvgPool 层
        for param in self.visual_backbone.parameters():
            param.requires_grad = False

        # 3. 融合层：简单的维度投影和拼接
        self.text_proj = nn.Linear(768, 256)
        self.vis_proj = nn.Linear(2048, 256)
        
        # 4. 分类器
        # 拼接后的维度是 256 + 256 = 512
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, bert_input_ids, bert_attention_mask, image_tensor, **kwargs):
        # 文本特征提取
        bert_out = self.bert(input_ids=bert_input_ids, attention_mask=bert_attention_mask)
        t_feat = bert_out.pooler_output # [batch, 768]
        t_feat = F.relu(self.text_proj(t_feat))
        
        # 图像特征提取
        v_feat = self.visual_backbone(image_tensor)
        v_feat = torch.flatten(v_feat, 1) # [batch, 2048]
        v_feat = F.relu(self.vis_proj(v_feat))
        
        # 核心逻辑：直接拼接（Late Fusion）
        # 这种做法无法识别“图文冲突”，只能识别“文字像假的”或“图片像假的”
        fused_feat = torch.cat([t_feat, v_feat], dim=1)
        
        logits = self.classifier(fused_feat)
        return logits

In [10]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(Dataset):
    def __init__(self, df, bert_tokenizer, max_len=128):
        self.df = df
        self.tokenizer = bert_tokenizer
        self.max_len = max_len
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text_clean"]) if pd.notna(row["text_clean"]) else ""
        img_path = row["img_local_path"]
        
        # 文本处理
        enc = self.tokenizer(text, padding='max_length', truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        
        # 图像处理
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.transform(image)
        except:
            pixel_values = torch.zeros((3, 224, 224))
            
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values": pixel_values,
            "label": torch.tensor(int(row["label"]), dtype=torch.long)
        }

In [11]:
import copy
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

def train_sccn():
    # 数据加载
    tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT_PATH)
    train_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv'), tokenizer), batch_size=32, shuffle=True)
    val_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/val_ready.csv'), tokenizer), batch_size=32)
    test_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv'), tokenizer), batch_size=32)

    model = LIIMR_Baseline(LOCAL_BERT_PATH).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # 稍微调低学习率以保证稳定性
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    save_path = "best_LIIMR_model.pth"

    for epoch in range(5):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            outputs = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["pixel_values"].to(DEVICE))
            loss = criterion(outputs, batch["label"].to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # 验证
        val_loss, val_acc, _, _ = evaluate_logic(model, val_loader, criterion, DEVICE)
        print(f"Epoch {epoch+1} Complete. Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved Best Model with Acc: {val_acc:.4f}")

    # 测试
    print("\n" + "="*20 + " Test Set Evaluation " + "="*20)
    model.load_state_dict(torch.load(save_path))
    evaluate_on_test_set(model, test_loader, criterion, DEVICE)

In [12]:
from sklearn.metrics import classification_report, accuracy_score

@torch.no_grad()
def evaluate_logic(model, dataloader, criterion, device):
    """用于训练过程中的轻量化验证"""
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    for batch in dataloader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(ids, mask, imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, all_labels, all_preds

@torch.no_grad()
def evaluate_on_test_set(model, test_loader, criterion, device):
    """用于最后输出详细的测试报告"""
    _, acc, labels, preds = evaluate_logic(model, test_loader, criterion, device)
    
    print(f"\nFinal Results on Test Set:")
    print(f"Test Acc: {acc:.6f}")
    print("-" * 60)
    # 输出 precision, recall, f1 等
    print(classification_report(labels, preds, target_names=['real', 'fake'], digits=6))

In [15]:
train_sccn()

Epoch 1:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 1 Complete. Val Loss: 0.6457, Val Acc: 0.8082
--> Saved Best Model with Acc: 0.8082


Epoch 2:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 2 Complete. Val Loss: 0.5515, Val Acc: 0.8202
--> Saved Best Model with Acc: 0.8202


Epoch 3:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 3 Complete. Val Loss: 0.4428, Val Acc: 0.8236
--> Saved Best Model with Acc: 0.8236


Epoch 4:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 4 Complete. Val Loss: 0.3700, Val Acc: 0.8459
--> Saved Best Model with Acc: 0.8459


Epoch 5:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 5 Complete. Val Loss: 0.3241, Val Acc: 0.8699
--> Saved Best Model with Acc: 0.8699

==================== Test Set Evaluation ====================

Final Results on Test Set:
Test Acc: 0.863986
------------------------------------------------------------
              precision    recall  f1-score   support

        real   0.866044  0.883943  0.874902       629
        fake   0.861480  0.840741  0.850984       540

    accuracy                       0.863986      1169
   macro avg   0.863762  0.862342  0.862943      1169
weighted avg   0.863936  0.863986  0.863853      1169



In [16]:
test_model = LIIMR_Baseline(LOCAL_BERT_PATH)
print(test_model)

LIIMR_Baseline(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af